## preprocess atac & rna data

In [1]:
import os 
import scanpy as sc 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import scb_multiome as scbm 

2025-08-12 18:18:16.322961: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.9.41). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


## download data

In [2]:
BASE_DIR = "/mnt/fxcai/nfs_share2"
db_data_path = f"../db_data"
raw_data_path = "./data/raw"

os.makedirs(db_data_path, exist_ok=True)
os.makedirs(raw_data_path, exist_ok=True)
scbm.data.download_jaspar_motifs(save_path=db_data_path)
scbm.data.download_encode_chip(save_path=db_data_path)      ## optional 
scbm.data.download_pbmc_raw(save_path=raw_data_path)        ## download h5 file from 10x  
scbm.data.download_pbmc_supp(save_path=raw_data_path)       ## download cell labels, cell type specific chip 

PosixPath('data/raw/pbmc_data_for_zenodo.tar.gz')

In [6]:
os.system(f"tar -xzf {db_data_path}/ENCODE_ChIP.tar.gz -C {db_data_path}")
os.system(f"tar -xzf {raw_data_path}/pbmc_data_for_zenodo.tar.gz -C {raw_data_path}")

0

In [7]:
data_path = f"{raw_data_path}/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5"
# data_path = f"{BASE_DIR}/Data/pbmc_10k/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5"

In [8]:
ad = sc.read_10x_h5(data_path, gex_only=False)
ad_rna = ad[:, ad.var['feature_types'] == "Gene Expression"]
ad_atac = ad[:, ad.var['feature_types'] == "Peaks"]

chr = ad_atac.var['gene_ids'].str.split(":").str[0]
pos = ad_atac.var['gene_ids'].str.split(":").str[1]
start, end = pos.str.split("-").str[0], pos.str.split("-").str[1]

ad_atac.var['chr'] = chr 
ad_atac.var['start'] = start 
ad_atac.var['end'] = end 

chrs = ['chr'+str(i) for i in range(1,23)] + ['chrX', 'chrY']
ad_atac = ad_atac[:, ad_atac.var['chr'].isin(chrs)]


/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/tmp/ipykernel_2831391/3167846094.py:9: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  ad_atac.var['chr'] = chr


In [9]:
### subset celltype based on LINGER annotation
celltype_path = f"{raw_data_path}/pbmc_data_for_zenodo/PBMC_label.txt"
# celltype = pd.read_csv(f"{BASE_DIR}/Data/pbmc_10k/LINGER_data/PBMC_label.txt", delimiter='\t', index_col=0)
celltype = pd.read_csv(celltype_path, delimiter='\t', index_col=0)
celltype

,label
barcode_use,
AAACAGCCAAGGAATC-1,naive CD4 T cells
AAACAGCCAATCCCTT-1,memory CD4 T cells
AAACAGCCAATGCGCT-1,naive CD4 T cells
AAACAGCCAGTAGGTG-1,naive CD4 T cells
AAACAGCCAGTTTACG-1,memory CD4 T cells
...,...
TTTGTTGGTGACATGC-1,naive CD8 T cells
TTTGTTGGTGTTAAAC-1,naive CD8 T cells
TTTGTTGGTTAGGATT-1,CD56 (bright) NK cells


In [5]:
ad_rna = ad_rna[celltype.index, :]
ad_atac = ad_atac[celltype.index, :]
ad_rna.obs['celltype'] = celltype
ad_rna

/tmp/ipykernel_162631/853283477.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  ad_rna.obs['celltype'] = celltype
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 9543 × 36601
    obs: 'celltype'
    var: 'gene_ids', 'feature_types', 'genome', 'interval'

In [6]:
# basic stats
sc.pp.filter_cells(ad_rna, min_genes=0)
sc.pp.filter_genes(ad_rna, min_cells=0)
sc.pp.filter_cells(ad_atac, min_genes=0)
sc.pp.filter_genes(ad_atac, min_cells=0)

# a gene need to be expressed in 0.5% cells
# a peak need to be accessible in 0.5% cells
thres = int(0.005*ad_rna.n_obs)
ad_rna = ad_rna[:, ad_rna.var['n_cells']>thres]
ad_atac = ad_atac[:, ad_atac.var['n_cells']>thres]

/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:167: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["n_genes"] = number


In [7]:
print(ad_rna.shape, ad_atac.shape)

(9543, 15449) (9543, 131511)


In [8]:
raw_count = ad_rna.X.copy()
sc.pp.normalize_total(ad_rna)
sc.pp.log1p(ad_rna)
ad_rna.layers['raw_count'] = raw_count

/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:207: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [9]:
import os 
os.makedirs("./data", exist_ok=True)
os.makedirs("./data/rna_data/", exist_ok=True)
os.makedirs("./data/atac_data/", exist_ok=True)

In [10]:

ad_rna.write_h5ad("./data/rna_data/ad_rna_15k.h5ad")
ad_atac.write_h5ad("./data/atac_data/ad_atac_132k.h5ad")

... storing 'celltype' as categorical


... storing 'feature_types' as categorical
... storing 'genome' as categorical
... storing 'interval' as categorical
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
... storing 'feature_types' as categorical
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
... storing 'genome' as categorical
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
... storing 'chr' as categorical
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWa

### extract tf expression

In [ ]:
import scb_multiome as scbm 

In [10]:
# jaspar_path = f"{BASE_DIR}/genomes/JASPAR_human_TFs_meme/20230424043428_JASPAR2022_combined_matrices_2028_meme.txt"  
jaspar_path = f"{db_data_path}/20230424043428_JASPAR2022_combined_matrices_2028_meme.txt"
df = scbm.pp.read_JASPAR_pwms(jaspar_path)
df

,motif,tf,pwm
0,MA0634.1,ALX3,"[[0.158817, 0.275993, 0.134823, 0.430367], [0...."
1,MA0007.2,AR,"[[0.560146, 0.099233, 0.272086, 0.068535], [0...."
2,MA1463.1,ARGFX,"[[0.234674, 0.105492, 0.269275, 0.39056], [0.1..."
3,MA0259.1,ARNT,"[[0.259615, 0.269231, 0.471154, 0.0], [0.09615..."
4,MA1464.1,ARNT2,"[[0.251527, 0.091059, 0.615761, 0.041654], [0...."
...,...,...,...
790,MA1720.1,ZNF85,"[[0.375, 0.17637, 0.246575, 0.202055], [0.3732..."
791,MA1721.1,ZNF93,"[[0.195561, 0.116834, 0.627722, 0.059883], [0...."
792,MA1602.1,ZSCAN29,"[[0.068068, 0.497497, 0.124124, 0.31031], [0.0..."
793,MA1722.1,ZSCAN31,"[[0.117787, 0.01222, 0.69699, 0.173003], [0.00..."


In [ ]:
tfs = df['tf'].unique()
print(len(tfs))
ad_rna = sc.read_h5ad("./data/rna_data/ad_rna_15k.h5ad")

ad_rna.X = ad_rna.layers['raw_count'].toarray()
ad_rna.var['gene_symbols'] = ad_rna.var.index.copy()
ad_rna_tf = ad_rna[:,ad_rna.var['gene_symbols'].isin(tfs)]
print(ad_rna_tf.shape)

631


/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


(9543, 322)


In [ ]:
## re-normalise 
raw_layer = ad_rna_tf.X.copy()

sc.pp.normalize_total(ad_rna_tf)
sc.pp.log1p(ad_rna_tf)
ad_rna_tf.layers['raw_count'] = raw_layer

/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:207: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [ ]:
ad_rna_tf.write("./data/rna_data/tfs.h5ad")

### split train/val/test cells 


In [ ]:
ad_rna_tf = sc.read_h5ad("./data/rna_data/tfs.h5ad")
scbm.pp.split_cells(n_cell=ad_rna_tf.n_obs,
                    write_dest = "./data/rna_data/cell_splits.h5")

In [12]:
### preprocess atac seq as in scbasset 

atac_dir = "./data/atac_data"
os.makedirs(f"{atac_dir}/scb_processed", exist_ok=True)
scbm.pp.scbasset_pp.preprocess_seqs(input_ad=f"{atac_dir}/ad_atac_132k.h5ad",
                                    input_fasta=f"{BASE_DIR}/genomes/hg38/hg38.fa",
                                    output_path=f"{atac_dir}/scb_processed")


successful writing h5ad file.
successful writing bed file.
successful writing split file.
successful writing sparse m.
process 0 peaks takes 3.5 s
process 1000 peaks takes 4.1 s
process 2000 peaks takes 4.6 s
process 3000 peaks takes 5.1 s
process 4000 peaks takes 5.6 s
process 5000 peaks takes 6.2 s
process 6000 peaks takes 6.7 s
process 7000 peaks takes 7.2 s
process 8000 peaks takes 7.8 s
process 9000 peaks takes 8.3 s
process 10000 peaks takes 8.8 s
process 11000 peaks takes 9.3 s
process 12000 peaks takes 9.9 s
process 13000 peaks takes 10.4 s
process 14000 peaks takes 10.9 s
process 15000 peaks takes 11.4 s
process 16000 peaks takes 12.0 s
process 17000 peaks takes 12.7 s
process 18000 peaks takes 13.5 s
process 19000 peaks takes 14.1 s
process 20000 peaks takes 14.6 s
process 21000 peaks takes 15.2 s
process 22000 peaks takes 15.8 s
process 23000 peaks takes 16.3 s
process 24000 peaks takes 16.8 s
process 25000 peaks takes 17.4 s
process 26000 peaks takes 17.9 s
process 27000 pe

## preprocess for benchmark (optional)
1. overlap atac peaks with chip data

#### encode chip data

In [11]:
# encode_dir = f"{BASE_DIR}/Data/ENCODE_ChIP/"
encode_dir = f"{db_data_path}/ENCODE_ChIP"
meta_data_path = f"{encode_dir}/ENCODE4_TF_ChIPseq_bed_list.tsv"
meta_data = pd.read_csv(meta_data_path, delimiter='\t')
meta_data.index = list(meta_data['bed narrowPeak'])

ad_atac = sc.read_h5ad("./data/atac_data/ad_atac_132k.h5ad")


In [ ]:
chip_bulk = scbm.pp.chip_atac_overlap(ad_atac=ad_atac, 
                                      encode_dir=f"{encode_dir}/bedNarrow_process", 
                                      meta_data=meta_data
                                      )


num of bed files 2389


  4%|▍         | 100/2389 [00:33<11:22,  3.35it/s]/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scb_multiome/preprocessing/chip_overlap.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  encode_overlap[f_accession] = atac_peaks.index.isin(overlap_df.index)
  4%|▍         | 101/2389 [00:33<11:04,  3.44it/s]/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scb_multiome/preprocessing/chip_overlap.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  encode_overlap[f_accession] = atac_pea

In [7]:
os.makedirs("./data/chip_data", exist_ok=True)
chip_bulk.T.to_csv(f"./data/chip_data/atac_132k_encode_byTF.csv")

#### cistrome chip data (ground truth for benchmark)

In [ ]:
ad_atac = sc.read_h5ad("./data/atac_data/ad_atac_132k.h5ad")
# chip_path = f"{BASE_DIR}/scb_multiome/pbmc_10k/LINGER/cistrome_data/blood_cistrome"
chip_path = f"{raw_data_path}/pbmc_data_for_zenodo/blood_cistrome"

meta_data = pd.read_csv(f"{chip_path}/download_info.txt", delimiter='\t', skiprows=9, index_col=0)
meta_data.index = meta_data.index.astype(str) + "_peaks"
meta_data['tf_ct'] = (meta_data['Transcription factor'] + "_" + meta_data['Cell type'])

gt = scbm.pp.chip_atac_overlap(ad_atac=ad_atac, 
                               encode_dir=chip_path,
                               meta_data=meta_data,
                               groupkey="tf_ct")

gt[:5]

num of bed files 20


100%|██████████| 20/20 [00:07<00:00,  2.57it/s]


,chr1:9790-10675,chr1:180599-181702,chr1:191168-192093,chr1:267565-268455,chr1:270876-271770,chr1:585752-586648,chr1:629499-630395,chr1:633580-634577,chr1:778283-779200,chr1:816877-817776,...,chrX:155768338-155769087,chrX:155819838-155820705,chrX:155841097-155841964,chrX:155872866-155873701,chrX:155874362-155875206,chrX:155880849-155881756,chrX:155891141-155892020,chrX:155966624-155967524,chrX:155997155-155998067,chrX:156030098-156030882
tf_ct,,,,,,,,,,,,,,,,,,,,,
CTCF_classical monocytes,False,False,False,True,False,True,True,False,True,False,...,False,True,False,False,False,False,False,False,False,False
CTCF_naive B cells,False,False,False,True,False,True,True,True,True,False,...,False,True,False,False,False,False,False,True,False,False
ETS1_naive CD4 T cells,False,False,False,False,False,False,True,True,True,False,...,False,False,False,False,False,True,False,False,False,False
FOXP3_naive CD4 T cells,False,False,False,False,False,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False
IRF1_classical monocytes,True,True,False,False,False,False,True,True,True,True,...,False,False,False,False,False,True,True,False,False,True


In [10]:
os.makedirs("./data/chip_data", exist_ok=True)
gt.T.to_csv(f"./data/chip_data/blood_gt.csv")

#### baseline metrics 
1. celltype specific spearmanr between peaks & tf expression
    - requires PeakVI for atac imputation, see https://docs.scvi-tools.org/en/stable/ for installation 
    - Processed data can directly be downloaded from [SpearmanR.csv](https://github.com/Cai-fx/scb_multiome_demo/tree/main/tutorials/pbmc_10k/data/spearmanR/SpearmanR.csv)

In [3]:
import scvi 

/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
ad_atac = sc.read_h5ad("./data/atac_data/ad_atac_132k.h5ad")
ad_rna = sc.read_h5ad("./data/rna_data/tfs.h5ad")
gt = pd.read_csv(f"./data/chip_data/blood_gt.csv", index_col=0)

In [6]:
### compute celltype specific SpearmanR 
scvi.model.PEAKVI.setup_anndata(ad_atac)
pvi = scvi.model.PEAKVI(ad_atac)
pvi.train()
prob_ac = pvi.get_accessibility_estimates(ad_atac)
ad_atac.layers['pvi'] = prob_ac.to_numpy()

Trainer will use only 1 of 3 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=3)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a b

Epoch 120/500:  24%|██▍       | 120/500 [16:26<52:03,  8.22s/it, v_num=1, train_loss_step=3.25e+6, train_loss_epoch=3.31e+8]
Monitored metric reconstruction_loss_validation did not improve in the last 50 records. Best score: 21319.434. Signaling Trainer to stop.


In [7]:
celltype_specific_SpearmanR = scbm.pp.celltype_specific_spearmanr(ad_atac=ad_atac,
                                                                  ad_rna=ad_rna,
                                                                  tfs_cts=gt.columns,
                                                                  atac_layer="pvi",
                                                                  n_jobs=16)

tf_idx =  202
number of cells in classical monocytes = 1848


  0%|          | 0/131511 [00:00<?, ?it/s]/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
100%|██████████| 131511/131511 [00:13<00:00, 10042.55it/s]


tf_idx =  202
number of cells in naive B cells = 282


  0%|          | 0/131511 [00:00<?, ?it/s]/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
100%|██████████| 131511/131511 [00:09<00:00, 14077.99it/s]


tf_idx =  150
number of cells in naive CD4 T cells = 1373


...

  0%|          | 0/131511 [00:00<?, ?it/s]/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
100%|██████████| 131511/131511 [00:11<00:00, 11439.51it/s]


tf_idx =  145
number of cells in classical monocytes = 1848


  0%|          | 0/131511 [00:00<?, ?it/s]/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
100%|██████████| 131511/131511 [00:12<00:00, 10798.51it/s]


tf_idx =  38
number of cells in classical monocytes = 1848


  0%|          | 0/131511 [00:00<?, ?it/s]/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
100%|██████████| 131511/131511 [00:12<00:00, 10670.80it/s]


In [8]:
celltype_specific_SpearmanR[:5]

,CTCF_classical monocytes,CTCF_naive B cells,ETS1_naive CD4 T cells,FOXP3_naive CD4 T cells,IRF1_classical monocytes,IRF4_naive B cells,MYC_naive B cells,REST_naive CD4 T cells,RUNX1_classical monocytes,RUNX1_myeloid DC,RUNX1_naive CD4 T cells,SPI1_classical monocytes,STAT1_classical monocytes
chr1:9790-10675,-0.037643,-0.074885,0.001815,0.039642,0.013194,0.141813,0.046560,-0.036721,0.039731,0.012173,-0.059391,-0.066867,-0.117094
chr1:180599-181702,-0.006186,-0.046487,0.022422,-0.015083,0.000941,0.023024,0.028424,0.006998,0.038440,-0.065074,-0.021464,-0.004569,-0.079383
chr1:191168-192093,-0.053476,0.020132,-0.016400,-0.034507,0.035849,0.037311,0.038512,-0.016399,0.028488,0.010008,-0.058862,-0.143144,-0.143588
chr1:267565-268455,-0.023655,-0.051345,0.037360,-0.091966,-0.032161,-0.043846,0.005317,0.043745,0.022623,0.089833,0.055974,-0.043534,-0.073145
chr1:270876-271770,-0.022037,0.006083,-0.030036,0.073670,-0.016850,0.052151,0.042568,-0.045158,0.063067,0.061070,-0.090555,0.026113,-0.035844


In [9]:
os.makedirs("./data/spearmanR", exist_ok=True)
celltype_specific_SpearmanR.to_csv("./data/spearmanR/SpearmanR.csv")

2. FIMO motif matching
- requires FIMO, install with ```conda install bioconda::meme```
- Processed data can directly be downloaded from [fimo_res.csv](https://github.com/Cai-fx/scb_multiome_demo/tree/main/tutorials/pbmc_10k/data/all_seq_match/fimo_res.csv) 

In [3]:
import h5py 

In [4]:
preprocess_folder_acc = f"./data/atac_data/scb_processed"
with h5py.File(f"{preprocess_folder_acc}/all_seqs.h5", "r") as f:
    all_seqs = f["X"][:]
ad_atac = sc.read_h5ad("./data/atac_data/ad_atac_132k.h5ad")


In [5]:
fasta_dir = f"./data/all_seq_match"
os.makedirs(fasta_dir, exist_ok=True)

if not os.path.exists(f"{fasta_dir}/all_seqs.fasta"):
    scbm.pp.write_fasta(f"{fasta_dir}/all_seqs.fasta", all_seqs, seq_names=ad_atac.var_names)

131511it [15:53, 137.99it/s]


In [ ]:
### run fimo for each motif in groundtruth
gt = pd.read_csv(f"./data/chip_data/blood_gt.csv", index_col=0)
# jaspar_path = f"{BASE_DIR}/genomes/JASPAR_human_TFs_meme/20230424043428_JASPAR2022_combined_matrices_2028_meme.txt"
jaspar_path = f"{db_data_path}/20230424043428_JASPAR2022_combined_matrices_2028_meme.txt"
motifs = scbm.pp.read_JASPAR_pwms(jaspar_path)
tfs = list(set([tf_ct.split("_")[0] for tf_ct in gt.columns]))
motifs= motifs[motifs["tf"].isin(tfs)]

mm = motifs['motif'].to_list()
mm

['MA1930.1',
 'MA0139.1',
 'MA1929.1',
 'MA0098.3',
 'MA0850.1',
 'MA0050.2',
 'MA1419.1',
 'MA0059.1',
 'MA0147.3',
 'MA0138.2',
 'MA0002.1',
 'MA0080.5',
 'MA0517.1',
 'MA0137.3']

In [6]:
seq_f = "./data/all_seq_match/all_seqs.fasta"
out_dir = "./data/all_seq_match/fimo_out"

os.makedirs(f"{out_dir}", exist_ok=True)
for motif in mm:
    fimo_cmd = f"""fimo --max-stored-scores 500000 --oc {out_dir}/{motif} --max-strand --motif {motif} {jaspar_path} {seq_f}"""
    os.system(fimo_cmd)



Skipping motif +MA0002.1.
Skipping motif -MA0002.1.
Skipping motif +MA0030.1.
Skipping motif -MA0030.1.
Skipping motif +MA0031.1.
Skipping motif -MA0031.1.
Skipping motif +MA0051.1.
Skipping motif -MA0051.1.
Skipping motif +MA0059.1.
Skipping motif -MA0059.1.
Skipping motif +MA0065.1.
Skipping motif -MA0065.1.
Skipping motif +MA0066.1.
Skipping motif -MA0066.1.
Skipping motif +MA0069.1.
Skipping motif -MA0069.1.
Skipping motif +MA0070.1.
Skipping motif -MA0070.1.
Skipping motif +MA0071.1.
Skipping motif -MA0071.1.
Skipping motif +MA0072.1.
Skipping motif -MA0072.1.
Skipping motif +MA0073.1.
Skipping motif -MA0073.1.
Skipping motif +MA0074.1.
Skipping motif -MA0074.1.
Skipping motif +MA0077.1.
Skipping motif -MA0077.1.
Skipping motif +MA0084.1.
Skipping motif -MA0084.1.
Skipping motif +MA0091.1.
Skipping motif -MA0091.1.
Skipping motif +MA0101.1.
Skipping motif -MA0101.1.
Skipping motif +MA0107.1.
Skipping motif -MA0107.1.
Skipping motif +MA0115.1.
Skipping motif -MA0115.1.
Skipping mot

In [ ]:
## read results with pval < 0.05, set other peaks' pval to 1. df contains log(pval)
fimo_res = scbm.pp.read_fimo_res(fimo_dir = out_dir, 
                                        peak_idx=ad_atac.var_names, 
                                        jaspar_motifs=motifs,
                                        f_name="fimo.txt",
                                        seq_name_key="sequence name")

fimo_res[:5]

/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scb_multiome/utils/IOs.py:91: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  fimo_res = pd.read_csv(file_name, sep="\t", skipfooter=skiprows)
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scb_multiome/utils/IOs.py:91: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  fimo_res = pd.read_csv(file_name, sep="\t", skipfooter=skiprows)
/mnt/fxcai/nfs_share2/anaconda3/envs/demo/lib/python3.9/site-packages/scb_multiome/utils/IOs.py:91: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  fimo_res = pd.read_csv(file_name, sep="\t", skipfooter=skiprows)
/mnt/fxcai/nfs_s

,CTCF:MA1930.1,CTCF:MA0139.1,CTCF:MA1929.1,ETS1:MA0098.3,FOXP3:MA0850.1,IRF1:MA0050.2,IRF4:MA1419.1,MYC:MA0059.1,MYC:MA0147.3,REST:MA0138.2,RUNX1:MA0002.1,SPI1:MA0080.5,STAT1:MA0517.1,STAT1:MA0137.3
chr1:9790-10675,0.000000,-10.129134,-9.572746,0.000000,0.0,0.000000,0.0,0.00000,0.0,0.000000,0.0,0.000000,0.000000,0.0
chr1:180599-181702,-10.298013,-20.430596,-31.791505,0.000000,0.0,0.000000,0.0,0.00000,0.0,0.000000,0.0,0.000000,0.000000,0.0
chr1:191168-192093,0.000000,0.000000,-10.490475,-10.328135,0.0,0.000000,0.0,0.00000,0.0,-10.780558,0.0,-9.624342,0.000000,0.0
chr1:267565-268455,0.000000,-12.848527,-15.166438,0.000000,0.0,-10.608707,0.0,0.00000,0.0,0.000000,0.0,-10.747458,-10.604667,0.0
chr1:270876-271770,-11.235294,-9.232586,0.000000,0.000000,0.0,-11.426748,0.0,-9.27969,0.0,-10.839981,0.0,0.000000,-9.277549,0.0


In [7]:
fimo_res.to_csv("./data/all_seq_match/fimo_res.csv")